In [1]:
!pip install --quiet open-clip-torch packaging ftfy regex tqdm "torch>=1.9.0" torchvision huggingface_hub safetensors timm transformers
!git clone --quiet https://github.com/UCSC-VLAA/CLIPS.git
!pip install --quiet -r CLIPS/requirements.txt

In [2]:
import torch
import torch.nn.functional as F
from transformers import CLIPProcessor, CLIPModel
from open_clip import create_model_from_pretrained, get_tokenizer
from PIL import Image

In [ ]:
image = Image.open("X:/Projects/CV/Assignment 3/human_dog.jpg").convert("RGB")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()

captions = [
    "A man walking a dog in the park",
    "A person sitting on a bench",
    "A lone tree against the sky",
    "A dog playing with a ball",
    "A man reading a newspaper",
    "Two dogs running together",
    "A child hugging a puppy",
    "A busy city street scene",
    "A dog sleeping indoors",
    "A man holding a leash"
]

clip_inputs = clip_processor(text=captions, images=image, return_tensors="pt", padding=True)

with torch.no_grad():
    clip_outputs = clip_model(**clip_inputs)
    clip_logits = clip_outputs.logits_per_image
    clip_probs = clip_logits.softmax(dim=-1)[0]

print("=== CLIP similarity scores ===")
for score, cap in zip(clip_probs.tolist(), captions):
    print(f"{score:0.4f}  —  {cap}")
    
clips_model, clips_preprocess = create_model_from_pretrained("hf-hub:UCSC-VLAA/ViT-L-14-CLIPS-224-Recap-DataComp-1B")
clips_tokenizer = get_tokenizer("hf-hub:UCSC-VLAA/ViT-L-14-CLIPS-224-Recap-DataComp-1B")
clips_model.eval()

clips_image = clips_preprocess(image).unsqueeze(0)
text_tokens = clips_tokenizer(captions, context_length=clips_model.context_length).to(clips_image.device)

with torch.no_grad(), torch.amp.autocast(device_type="cuda"):
    image_features = clips_model.encode_image(clips_image)
    text_features  = clips_model.encode_text(text_tokens)
    image_features = F.normalize(image_features, dim=-1)
    text_features  = F.normalize(text_features, dim=-1)
    clips_logits   = (100.0 * image_features @ text_features.T).softmax(dim=-1)[0]

print("\n=== CLIPS similarity scores ===")
for score, cap in zip(clips_logits.tolist(), captions):
    print(f"{score:0.4f}  —  {cap}")

=== CLIP similarity scores ===
0.0779  —  A man walking a dog in the park
0.0014  —  A person sitting on a bench
0.0000  —  A lone tree against the sky
0.0032  —  A dog playing with a ball
0.0031  —  A man reading a newspaper
0.0220  —  Two dogs running together
0.0169  —  A child hugging a puppy
0.0001  —  A busy city street scene
0.0335  —  A dog sleeping indoors
0.8420  —  A man holding a leash

=== CLIPS similarity scores ===
0.0039  —  A man walking a dog in the park
0.0000  —  A person sitting on a bench
0.0000  —  A lone tree against the sky
0.2376  —  A dog playing with a ball
0.0039  —  A man reading a newspaper
0.0030  —  Two dogs running together
0.2794  —  A child hugging a puppy
0.0002  —  A busy city street scene
0.0017  —  A dog sleeping indoors
0.4702  —  A man holding a leash


**CLIP vs. CLIPS similarity scores**

- **Peakiness**  
  - **CLIP** concentrates most of its probability on one caption (“A man holding a leash” ≈ 84%), with almost zero on others.  
  - **CLIPS** spreads its mass across several plausible captions (top three sum to ≈ 99%), indicating a softer distribution.

- **Synonym Robustness**  
  - CLIP strongly favors the exact best‐matched phrase, underweighting related descriptions (“walking a dog” ≈ 7.8%).  
  - CLIPS gives substantial weight to semantically similar captions (“playing with a ball” ≈ 23.8%, “hugging a puppy” ≈ 27.9%), showing greater paraphrase tolerance.

- **Confidence vs. Coverage**  
  - CLIP’s high confidence in one label is useful for a single definitive prediction.  
  - CLIPS’s broader attention to multiple correct captions is advantageous when retrieving or ranking several valid descriptions.

- **Practical Implications**  
  - Use **CLIP** when you need one clear answer.  
  - Use **CLIPS** when you want a more nuanced, multi‐candidate retrieval (e.g., caption augmentation or ensemble filtering).
